In [4]:
import pandas as pd

In [5]:
splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}

In [6]:
df_test = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["test"])

In [7]:
df_test_exp = df_test.iloc[0:30,:]

In [1]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [2]:
import re
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = "auto",
    device_map = "auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
import re

def extract_answer(text):
    """
    Extracts numerical digits from model responses.
    Handles '#### 72', 'Final Answer: ... 72 clips.', and trailing digits.
    """
    if not text:
        return None

    # 1. Standard GSM8K format: "#### 72" or "#### $72"
    match = re.search(r"####\s*\$?\s*([0-9,\.]+)", text)
    if match:
        ans = match.group(1).rstrip('.').replace(",", "")
        return ans

    # 2. Look after "Final Answer:" for the FIRST sequence of digits
    match_heading = re.search(r"final\s*answer[:\s]*.*?\b([0-9]+(?:\.[0-9]+)?)\b", text, re.IGNORECASE)
    if match_heading:
        return match_heading.group(1)

    # 3. Fallback: Find all sequences of pure digits in the text and return the last one
    numbers = re.findall(r"\b([0-9]+(?:\.[0-9]+)?)\b", text)
    if numbers:
        return numbers[-1]

    return None

In [9]:
def generate_response(messages, do_sample = False, temperature=0.7, top_p=0.9, max_new_tokens=512):

  """

  """


  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt = True
  )


  model_inputs = tokenizer([text], return_tensors = "pt").to(model.device)

  #configure generation parameters

  gen_kwargs = {
      "max_new_tokens": max_new_tokens,
      "do_sample": do_sample,
  }
  if do_sample:
    gen_kwargs["temperature"]= temperature
    gen_kwargs["top_p"] = top_p

  with torch.no_grad():
    output_ids = model.generate(**model_inputs, **gen_kwargs)

  #strip prompt tokens from output
  input_lenght = model_inputs.input_ids.shape[1]
  generated_tokens = output_ids[0][input_lenght:]

  num_generated_tokens = len(generated_tokens)
  response_text = tokenizer.decode(generated_tokens, skip_special_tokens = True)

  return response_text, num_generated_tokens

In [10]:
def build_four_shot_prompt(few_shot_df, current_question):
    """
    Constructs the few-shot context from df_shot and appends the target question.
    """
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Solve the math problem step by step, following the format of the examples. At the end, state the final numerical answer after '####'."}
    ]

    # Add the 4 demonstration examples as alternating user/assistant turns (or inside a single user prompt)
    # Using a single structured user prompt format:
    few_shot_text = "Here are some examples of how to solve math word problems:\n\n"
    for _, row in few_shot_df.iterrows():
        few_shot_text += f"Question: {row['question']}\nAnswer:\n{row['answer']}\n\n"

    few_shot_text += f"Now solve this new problem:\nQuestion: {current_question}\nAnswer:"

    messages.append({"role": "user", "content": few_shot_text})
    return messages

In [11]:
import ast
import math
import operator

# Supported binary operators
SAFE_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

# Supported mathematical functions
SAFE_FUNCTIONS = {
    "abs": abs,
    "round": round,
    "sqrt": math.sqrt,
    "ceil": math.ceil,
    "floor": math.floor,
}

def restricted_calculator(expression_str: str) -> str:
    """
    Safely evaluates basic mathematical expressions using AST.
    Returns the string result or a descriptive error message.
    """
    clean_expr = expression_str.strip().replace("$", "").replace(",", "")

    try:
        node = ast.parse(clean_expr, mode='eval').body

        def _eval_node(n):
            if isinstance(n, ast.Num):  # Literal numbers
                return n.n
            elif isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
                return n.value
            elif isinstance(n, ast.BinOp):  # Binary operations (+, -, *, /...)
                left = _eval_node(n.left)
                right = _eval_node(n.right)
                op_type = type(n.op)
                if op_type in SAFE_OPERATORS:
                    return SAFE_OPERATORS[op_type](left, right)
                raise ValueError(f"Unsupported operator: {op_type.__name__}")
            elif isinstance(n, ast.UnaryOp):  # Unary operations (-x, +x)
                operand = _eval_node(n.operand)
                op_type = type(n.op)
                if op_type in SAFE_OPERATORS:
                    return SAFE_OPERATORS[op_type](operand)
                raise ValueError(f"Unsupported unary operator: {op_type.__name__}")
            elif isinstance(n, ast.Call):  # Allowed math functions
                if isinstance(n.func, ast.Name) and n.func.id in SAFE_FUNCTIONS:
                    args = [_eval_node(arg) for arg in n.args]
                    return SAFE_FUNCTIONS[n.func.id](*args)
                raise ValueError(f"Function call not allowed or unrecognized: {ast.dump(n.func)}")
            else:
                raise ValueError(f"Disallowed construct: {type(n).__name__}")

        result = _eval_node(node)
        return str(result)

    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Error: Invalid expression ({str(e)})"

In [13]:
import re
import pandas as pd

df_columns = [
    "Method",
    "Question Number",
    "Model Answer",
    "Correct Answer",
    "False",
    "True",
    "Total Generated Token"
]

AGENT_SYSTEM_PROMPT = (
    "You are a math reasoning agent with access to a Python calculator tool.\n"
    "To solve word problems:\n"
    "1. Break down the problem step by step.\n"
    "2. Whenever you need to perform arithmetic, write exact math expressions using the syntax:\n"
    "   CALCULATE: <expression>\n"
    "3. You will receive an OBSERVATION with the result or error. Use it for your next step.\n"
    "4. You may call CALCULATE at most 3 times.\n"
    "5. Once you have the final answer, state it clearly after '#### ' (e.g., #### 42)."
)

agent_rows = []
print("=== METHOD 5: TOOL-INTEGRATED REACT CALCULATOR AGENT ===")

for i in range(len(df_test_exp)):
    q_text = df_test_exp.iloc[i, 0]
    true_ans_raw = df_test_exp.iloc[i, 1]
    true_ans = extract_answer(true_ans_raw)

    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": q_text}
    ]

    total_tokens_q = 0
    calc_calls_made = 0
    max_calc_calls = 3
    final_response = ""

    # Multi-turn interaction loop
    while calc_calls_made <= max_calc_calls:
        # Generate model turn
        resp_text, token_count = generate_response(messages, do_sample=False)
        total_tokens_q += token_count
        final_response = resp_text

        # Check if model triggered a calculation command
        calc_match = re.search(r"CALCULATE:\s*(.+)", resp_text)
        final_ans_match = "####" in resp_text

        if calc_match and calc_calls_made < max_calc_calls and not final_ans_match:
            expr_to_calc = calc_match.group(1).split("\n")[0].strip()
            obs_result = restricted_calculator(expr_to_calc)
            calc_calls_made += 1

            # Append model turn + tool observation into ongoing conversation context
            messages.append({"role": "assistant", "content": resp_text})
            messages.append({"role": "user", "content": f"OBSERVATION: {obs_result}"})
        else:
            # Reached final answer or exhausted calculator calls
            break

    model_ans = extract_answer(final_response)
    is_correct = (model_ans == true_ans) if (model_ans is not None and true_ans is not None) else False

    # Record structured output matching previous DataFrame formats
    agent_rows.append({
        "Method": "Tool-Agent",
        "Question Number": i + 1,
        "Model Answer": model_ans,
        "Correct Answer": true_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": total_tokens_q
    })

    print(12*"-", f"Question {i+1} (ReAct Agent)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* Calculator Calls Made: {calc_calls_made}")
    print(f"* Final Extracted Answer: {model_ans} | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Total Tokens ({calc_calls_made + 1} generation turns): {total_tokens_q}\n\n")

dfAgent = pd.DataFrame(agent_rows, columns=df_columns)

=== METHOD 5: TOOL-INTEGRATED REACT CALCULATOR AGENT ===
------------ Question 1 (ReAct Agent) ------------
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* Calculator Calls Made: 0
* Final Extracted Answer: 80 | Correct: False
Correct Answer: 18
Total Tokens (1 generation turns): 5


------------ Question 2 (ReAct Agent) ------------
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

* Calculator Calls Made: 0
* Final Extracted Answer: 4 | Correct: False
Correct Answer: 3
Total Tokens (1 generation turns): 4


------------ Question 3 (ReAct Agent) ------------
Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This i

In [ ]:
fshot_rows = []

print("=== METHOD 2: FOUR-SHOT PROMPTING ===")

for i in range(df_test_exp.shape[0]):
    q_text = df_test_exp.iloc[i, 0]
    true_ans_raw = df_test_exp.iloc[i, 1]

    messages = build_four_shot_prompt(df_test_exp, q_text)
    response_text, token_count = generate_response(messages, do_sample=False)

    model_ans = extract_answer(response_text)
    correct_ans = extract_answer(true_ans_raw)

    is_correct = (model_ans == correct_ans) if (model_ans is not None and correct_ans is not None) else False

    fshot_rows.append({
        "Method": "Four-Shot",
        "Question Number": i + 1,
        "Model Answer": model_ans,
        "Correct Answer": correct_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": token_count
    })

    print(12*"-", f"Question {i+1}", 12*"-", "\n", q_text)
    print()
    print(f"* MODEL's Response-{i+1}: \n\n** ", response_text, " **", "\n\n\n", sep="")
    print()
    print(f"* MODEL's Answer-{i+1}: \n\n** ", model_ans, " **", "\n", sep="")
    print("Correct Answer :", correct_ans, "\n")
    print(f"Generated Tokens for Response-{i+1} :", token_count, "\n\n")

dfFshot = pd.DataFrame(fshot_rows, columns=df_columns)

=== METHOD 2: FOUR-SHOT PROMPTING ===
------------ Question 1 ------------ 
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* MODEL's Response-1: 

** To determine how much Janet makes at the farmers' market each day, let's break down the information given:

1. **Daily Egg Production**:
   - Janet's ducks lay 16 eggs per day.

2. **Egg Sales**:
   - She sells 16 eggs per day.

3. **Eggs Sold Before Breakfast**:
   - She eats 3 eggs per day.
   - Therefore, the number of eggs remaining after breakfast is \(16 - 3 = 13\).

4. **Eggs Sold During Breakfast**:
   - She bakes muffins for 2 days.
   - So, the number of muffins baked is \(2 \times 3 = 6\).
   - Therefore, the number of eggs remaining after baking muffins is \(13 - 6 = 7\).

5. **Total Eggs Sold**

In [ ]:
# Assuming dfSelfC was the strongest method from Question 1
df_comparison = pd.concat([dfFshot, dfAgent], ignore_index=True)

metrics_summary = df_comparison.groupby("Method").agg(
    Total_Questions=("Question Number", "count"),
    Correct_Count=("True", "sum"),
    Accuracy_Pct=("True", lambda x: (x.sum() / x.count()) * 100),
    Avg_Tokens_Per_Q=("Total Generated Token", "mean"),
    Total_Tokens=("Total Generated Token", "sum")
).reset_index()

print("=== COMPARATIVE EVALUATION SUMMARY ===")
print(metrics_summary)